<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiazlization

In [1]:
# Params
VERBOSE = True
CHOSEN = 'gpt-oss'
FILE_NAMES = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged','7_wonders']
IT = 5
BASE_URL = 'https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/flawed_texts/'
OVERWRITE = ['']
try:
    #DRIVE
    from google.colab import drive
    drive.mount('/content/drive',force_remount=True)
    BASE_FOLDER = 'drive/MyDrive/NLP_proj/'
except:
    #LOCAL
    BASE_FOLDER = './'

In [3]:
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
from huggingface_hub import InferenceClient
import requests
import json
import torch
import os
with open(f'{BASE_FOLDER}/hf_api_key', 'r') as f:
    api_key = f.read()


model = InferenceClient(
    model="openai/gpt-oss-120b",
    api_key=api_key[:-1],
)

def generate_message(sys_prompt, usr_prompt):
    return [
        {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": usr_prompt,
            },
    ]

# Error detection

 Each rulebook is edited by inserting errors that are increasingly difficult to spot:
 - level 0: **original**    -> unaltered rulebook
 - level 1: **missing**     -> an entire paragraph of the rulebook describing some core mechanic is missing
- level 2: **unsolvable**   -> a rule directly contradicts another one
- level 3: **incoherent**   -> a combination of rules hardlocks the game
- level 4: **gamebreaking** -> a coherent but obviously unbalanced mechanic

In [7]:
prompt = """You are an expert board-game player.
Examine the rulebook provided by the user.
Proceed with a chain of thought:

- Scan the text linearly and note any statements that conflict with earlier ones.
- For each of the game mechanic check if it is explained.
- Check whether any mechanic could halt the game or give a player an overwhelming advantage.

If you encounter one or more problem, report the **most impactful** one quoting the relevant line and summarizing its impact.
If nothing stands out, reply “The rules appear consistent.”
"""



output_dict = {'lvl'+str(lvl): {str(it): '' for it in range(IT)} for lvl in range(5)}
to_do = [g for g in FILE_NAMES if f'{CHOSEN}_{g}.json' not in os.listdir(os.path.join(BASE_FOLDER,'error_detection'))
                                 or f'{CHOSEN}_{g}.json' in OVERWRITE]

if to_do == []:
    print('Nothing to do')

for g in to_do:
    for it,lvl in tqdm([(it,lvl) for it in range(IT) for lvl in range(5)]):
        rulebook = requests.get(BASE_URL+'lvl'+str(lvl)+'/'+g+'.txt').text #rulebook is cached so should not be a problem
        name = g.replace('_',' ')
        out = model.chat_completion(generate_message(prompt, "Here is the full rulebook of the game {name}:\n"+rulebook),
                                           temperature=0.7).choices[0].message.content
        output_dict['lvl'+str(lvl)][str(it)] = out
        if(VERBOSE):
            print(f'{g} lvl{lvl}, iteration: {it}\n---------\n{out}')


    with open(f'{BASE_FOLDER}error_detection/{CHOSEN}_{g}.json','w') as f:
        json.dump(dict(output_dict),f)

  4%|█▊                                          | 1/25 [00:03<01:15,  3.16s/it]

ticket_to_ride lvl0, iteration: 0
---------
**Most impactful problem – the ticket‑scoring values are never defined**

> “Shuffle the tickets and deal 4 cards to each player. … Any returned cards are shuffled together and put at the bottom of the ticket deck …”  
> …  
> “Players reveal all tickets they kept. The value of successfully completed tickets is added to their total score. The value of any incomplete tickets is deducted from their total score.”  

The rulebook never explains **how many points each ticket is worth** (or how negative points are calculated for failed tickets). The only place ticket points appear is in an example (“Blue managed to complete both the ‘Montréal – Atlanta’ and ‘New York – Atlanta’ tickets, adding 15 points…”) but there is no general rule stating, for instance, “each ticket shows its point value on the card” or “ticket values range from 1‑20 points as printed on the card”.  

**Impact**  
Without a rule that tells players what a ticket’s point value is

  8%|███▌                                        | 2/25 [00:07<01:28,  3.84s/it]

ticket_to_ride lvl1, iteration: 0
---------
**Most impactful problem – the core mechanic “Claim a route” is never explained.**  

- The rulebook lists “Claim 1 route” as one of the three possible actions a player can take on their turn, but there is **no description** of how a route is claimed (how many and what colour train cards are required, what to do with double‑routes, tunnels, ferries, etc.) and **no scoring table** showing how many points a claimed route is worth.  

> *“On your turn, you must perform 1 (and only 1) of the following 3 actions:  
> 1. Draw train cards  
> 2. Claim 1 route  
> 3. Draw tickets”*  

> *(No further rules are given for “Claim 1 route.”)*  

**Impact**  
Without any guidance on the claim‑route process, players cannot:

1. Determine what cards they must spend to take a route.  
2. Know how many plastic trains to place or whether a route is even legal to claim.  
3. Calculate the points earned from the route, which is essential for scoring throughout th

 12%|█████▎                                      | 3/25 [00:10<01:20,  3.66s/it]

...

7_wonders lvl2, iteration: 4
---------
**Most impactful problem – not enough Military Victory tokens for a full game**

> **“• 48 Military Conflict tokens: 24 Defeats and 24 Victories (8 per Age)”**  
> (Contents list)

> **Resolution of Military Conflicts**  
> “If your military strength is higher, take the **Military Victory token that corresponds to the current Age**.”  
> (Age I → +1 token, Age II → +3 token, Age III → +5 token)

In a game with the maximum number of players (7), each player has **two neighbours**.  
During a single Age a player can therefore earn **up to two Victory tokens** (one from each neighbour).  

- With 7 players the total possible Victory tokens needed in one Age = 7 players × 2 tokens = 14 tokens.  
- The rules supply only **8 Victory tokens per Age** (24 Victory tokens ÷ 3 Ages).

Consequences  

1. **Token shortage** – after the first few conflicts the Victory‑token supply for that Age will be exhausted, leaving players unable to receive the promised re

 96%|█████████████████████████████████████████▎ | 24/25 [01:35<00:04,  4.29s/it]

7_wonders lvl3, iteration: 4
---------
**Most impactful problem – the “extra turn” rule in the Military Conflict resolution can halt the game (or create an endless loop).**

**Quote from the rulebook (Resolution of Military Conflicts):**  

> “If your military strength is lower, **play another turn of the current age dealing 1 card to each player. After this repeat the resolution of military conflicts step**.”

**Why this is a problem**

1. **Age structure is broken** – Each Age is defined to have exactly **six turns** (the hand shrinks from 7 cards to 2, then one is discarded). The rule above allows an undefined number of additional turns, contradicting the “six‑turn” framework introduced earlier.

2. **No cards may be available** – By the end of the sixth turn every player has only two cards left, one of which is discarded and the other is played. After the normal six turns the draw piles are exhausted. If a player still has lower military strength, the rule forces “another turn” eve

100%|███████████████████████████████████████████| 25/25 [01:40<00:00,  4.01s/it]

7_wonders lvl4, iteration: 4
---------
**Most impactful problem – contradictory instruction on the passing direction**

> **“3. PASS YOUR HAND** … *give them to the neighbor on your left (during Ages I and III) or to the neighbor on your right (during Age II). The direction for passing is shown on the back of the Age cards.*”  

The rule gives **two different sources of truth** for how cards are passed between players:

1. **Explicit rule** – left in Ages I & III, right in Age II.  
2. **“Direction shown on the back of the Age cards.”**  

If the backs of the Age cards indicate a different direction (or if a player looks at the back first and follows it), the game will diverge from the stated left/right rule. Because passing determines which cards each player will see in subsequent turns, a mis‑pass can completely alter resource availability, building chains, and the timing of military conflicts. In a game that relies on simultaneous play and a fixed flow of hands, this ambiguity can h